In [10]:
import pandas as pd
import pyarrow as pa
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import IdentityTransform, MonthTransform
from pyiceberg.table.sorting import SortOrder, SortField
from pyiceberg.catalog import load_catalog

catalog = load_catalog(
    "default",
    **{
        "type": "hive",
        "uri": "thrift://hive-metastore:9083",
        "s3.endpoint": "http://minio:9000",
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin",
        "s3.use-ssl": "false",
        "s3.region-name": "us-east-1",
    },
)

In [11]:
# 1. Cek daftar namespace yang sudah ada
namespaces = catalog.list_namespaces()
print(f"Namespace saat ini: {namespaces}")

# 2. Buat namespace 'silver' jika belum ada
tgt_namespace = "silver"
loc_namespace = f"s3a://lakehouse/{tgt_namespace}/"

if any(tgt_namespace in ns for ns in namespaces):
    print(f"ℹ️ Namespace '{tgt_namespace}' sudah tersedia.")
else:
    catalog.create_namespace(tgt_namespace, properties={"location": loc_namespace})
    print(f"✅ Namespace '{tgt_namespace}' berhasil dibuat.")

Namespace saat ini: [('bronze',), ('default',), ('silver',)]
ℹ️ Namespace 'silver' sudah tersedia.


In [12]:
# 1. Load tabel Bronze dan ambil skemanya
table_bronze = catalog.load_table("bronze.yellow_taxi")
iceberg_schema = table_bronze.schema()
arrow_table = table_bronze.scan().to_arrow()
print(f"✅ Berhasil memuat {len(arrow_table)} baris data.")

# 2. Ambil Field ID untuk VendorID secara dinamis
# Ini kunci agar tidak kena ValueError lagi
vendor_id_field = iceberg_schema.find_field("VendorID")

# 3. Buat Partition Spec Sederhana (Hanya VendorID)
spec = PartitionSpec(
    PartitionField(
        source_id=vendor_id_field.field_id,
        field_id=1000, 
        transform=IdentityTransform(),
        name="vendor_id_partition"
    )
)

sort_order = SortOrder(
    SortField(
        source_id=iceberg_schema.find_field("tpep_pickup_datetime").field_id,
        transform=IdentityTransform()
    )
)

✅ Berhasil memuat 3066766 baris data.


In [13]:
schema_name = "silver"
table_name = "yellow_taxi"
table_id = f"{schema_name}.{table_name}"

table_location = f"s3a://lakehouse/{schema_name}/{table_name}"

try:
    # 1. Coba buat tabel baru
    table = catalog.create_table(
        identifier=table_id,
        schema=iceberg_schema,
        location=table_location,
        partition_spec=spec,
        sort_order=sort_order
    )
    print(f"✅ Tabel {table_id} berhasil dibuat.")
except Exception:
    # 2. Jika sudah ada, load tabel yang sudah ada
    table = catalog.load_table(table_id)
    print(f"ℹ️ Tabel {table_id} sudah ada, memuat tabel...")

ℹ️ Tabel silver.yellow_taxi sudah ada, memuat tabel...


In [14]:
# 3. Masukkan data (Append)
table.overwrite(arrow_table)
print(f"🚀 Data berhasil dimuat ke {table_id}!")

🚀 Data berhasil dimuat ke silver.yellow_taxi!
